In [ ]:
# ============================================================
# 02_pre_bi.ipynb
# Objetivo: preparar dataset final para o Power BI (fato + dims)
# Fonte principal: datamart_wide_aluno_ano.csv
# ============================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
# ------------------------------------------------------------
# 1) Lendo a base "wide" (1 linha por aluno-ano)
# ------------------------------------------------------------

PATH_WIDE = "datamart_wide_aluno_ano.csv"

df = pd.read_csv(PATH_WIDE)

print("Shape:", df.shape)
display(df.head())
display(df.dtypes)


Shape: (3030, 18)


,RA,NOME,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,ANO_INGRESSO,IAN,IDADE,IEG,IAA,IPS,IPV,INDE,PEDRA,ANO,IPP,DEFASAGEM
0,RA-1,Aluno-1,7,A,Menina,Escola Pública,2016,5.0,19,4.1,8.3,5.6,7.278,5.783,Quartzo,2022,NaN,NaN
1,RA-2,Aluno-2,7,A,Menina,Rede Decisão,2017,10.0,17,5.2,8.8,6.3,6.778,7.055,Ametista,2022,NaN,NaN
2,RA-3,Aluno-3,7,A,Menina,Rede Decisão,2016,10.0,17,7.9,0.0,5.6,7.556,6.591,Ágata,2022,NaN,NaN
3,RA-4,Aluno-4,7,A,Menino,Rede Decisão,2017,10.0,17,4.5,8.8,5.6,5.278,5.951,Quartzo,2022,NaN,NaN
4,RA-5,Aluno-5,7,A,Menina,Rede Decisão,2016,10.0,17,8.6,7.9,5.6,7.389,7.427,Ametista,2022,NaN,NaN


,0
RA,object
NOME,object
FASE,object
TURMA,object
GENERO,object
INSTITUICAO_DE_ENSINO,object
ANO_INGRESSO,int64
IAN,float64
IDADE,object
IEG,float64


In [ ]:
# ------------------------------------------------------------
# 2) Padronizando tipos (ANO numérico, indicadores como float)
# ------------------------------------------------------------

# garante ANO como int (quando possível)
if "ANO" in df.columns:
    df["ANO"] = pd.to_numeric(df["ANO"], errors="coerce").astype("Int64")

# colunas numéricas típicas (se existirem)
indicadores_num = ["INDE", "IAA", "IAN", "IDA", "IEG", "IPP", "IPS", "IPV", "DELTA_INDE"]
for c in indicadores_num:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# dimensões comuns (deixa como string pra evitar problemas)
dims_texto = ["RA", "NOME", "GENERO", "FASE", "TURMA", "INSTITUICAO_DE_ENSINO"]
for c in dims_texto:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip()

display(df.head())


,RA,NOME,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,ANO_INGRESSO,IAN,IDADE,IEG,IAA,IPS,IPV,INDE,PEDRA,ANO,IPP,DEFASAGEM
0,RA-1,Aluno-1,7,A,Menina,Escola Pública,2016,5.0,19,4.1,8.3,5.6,7.278,5.783,Quartzo,2022,NaN,NaN
1,RA-2,Aluno-2,7,A,Menina,Rede Decisão,2017,10.0,17,5.2,8.8,6.3,6.778,7.055,Ametista,2022,NaN,NaN
2,RA-3,Aluno-3,7,A,Menina,Rede Decisão,2016,10.0,17,7.9,0.0,5.6,7.556,6.591,Ágata,2022,NaN,NaN
3,RA-4,Aluno-4,7,A,Menino,Rede Decisão,2017,10.0,17,4.5,8.8,5.6,5.278,5.951,Quartzo,2022,NaN,NaN
4,RA-5,Aluno-5,7,A,Menina,Rede Decisão,2016,10.0,17,8.6,7.9,5.6,7.389,7.427,Ametista,2022,NaN,NaN


In [ ]:
# ------------------------------------------------------------
# 3) Padronização de valores categóricos (visão de negócio)
# ------------------------------------------------------------

# Exemplo: normalizar gênero para algo consistente no BI
if "GENERO" in df.columns:
    mapa_genero = {
        "Menina": "Feminino",
        "Menino": "Masculino",
        "Feminino": "Feminino",
        "Masculino": "Masculino",
        "F": "Feminino",
        "M": "Masculino"
    }
    df["GENERO"] = df["GENERO"].replace(mapa_genero)

# Padroniza instituição: remove espaços duplos e padroniza caixa
if "INSTITUICAO_DE_ENSINO" in df.columns:
    df["INSTITUICAO_DE_ENSINO"] = (
        df["INSTITUICAO_DE_ENSINO"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

display(df[["RA","ANO","GENERO","INSTITUICAO_DE_ENSINO"]].head(10))


,RA,ANO,GENERO,INSTITUICAO_DE_ENSINO
0,RA-1,2022,Feminino,Escola Pública
1,RA-2,2022,Feminino,Rede Decisão
2,RA-3,2022,Feminino,Rede Decisão
3,RA-4,2022,Masculino,Rede Decisão
4,RA-5,2022,Feminino,Rede Decisão
5,RA-6,2022,Feminino,Escola Pública
6,RA-7,2022,Masculino,Rede Decisão
7,RA-8,2022,Feminino,Escola Pública
8,RA-9,2022,Feminino,Escola Pública
9,RA-10,2022,Feminino,Escola Pública


In [ ]:
# ------------------------------------------------------------
# 4) Checagens de integridade (RA + ANO deve ser único)
# ------------------------------------------------------------

ID_COL = "RA"

dup = df.duplicated(subset=[ID_COL, "ANO"]).sum()
print("Duplicados (RA, ANO):", dup)

if dup > 0:
    display(
        df[df.duplicated(subset=[ID_COL, "ANO"], keep=False)]
        .sort_values([ID_COL, "ANO"])
        .head(50)
    )


Duplicados (RA, ANO): 0


In [ ]:
# ------------------------------------------------------------
# 5) Recalcular DELTA_INDE de forma estável (ano a ano por aluno)
#    ⚠️ NÃO arredondar para 3 casas aqui, senão você “mata” variações pequenas.
# ------------------------------------------------------------

if "INDE" in df.columns:
    # garante INDE numérico (se vier como texto com vírgula, também funciona)
    df["INDE"] = (
        df["INDE"]
        .astype("string")
        .str.replace(",", ".", regex=False)
    )
    df["INDE"] = pd.to_numeric(df["INDE"], errors="coerce")

    df = df.sort_values([ID_COL, "ANO"])

    # delta bruto (sem arredondar)
    df["DELTA_INDE_CALC"] = df.groupby(ID_COL)["INDE"].diff()

    # opcional: arredondar com mais precisão (6 casas) para manter detalhe
    df["DELTA_INDE_CALC"] = df["DELTA_INDE_CALC"].round(6)

    # remove o efeito visual de -0.0 e valores minúsculos
    df["DELTA_INDE_CALC"] = df["DELTA_INDE_CALC"].mask(df["DELTA_INDE_CALC"].abs() < 1e-6, 0)

    # sobrescreve DELTA_INDE final (é o que o Power BI vai consumir)
    df["DELTA_INDE"] = df["DELTA_INDE_CALC"]

display(df[[ID_COL, "ANO", "INDE", "DELTA_INDE"]].head(30))


,RA,ANO,INDE,DELTA_INDE
0,RA-1,2022,5.783,0.0
1855,RA-1,2023,<NA>,0.0
2958,RA-1,2024,<NA>,0.0
9,RA-10,2022,5.784,0.0
99,RA-100,2022,7.618,0.0
1072,RA-1000,2023,7.9162,0.0
2225,RA-1000,2024,8.364791,0.448591
1074,RA-1001,2023,8.1162,0.0
2227,RA-1001,2024,7.796432,-0.319768
1075,RA-1002,2023,7.9012,0.0


In [ ]:
# ------------------------------------------------------------
# 6) Features de negócio (faixas e flags)
# ------------------------------------------------------------

# garante que DELTA_INDE é numérico
if "DELTA_INDE" in df.columns:
    df["DELTA_INDE"] = pd.to_numeric(df["DELTA_INDE"], errors="coerce")

    # Flag: aluno piorou (delta negativo)
    # usa int64 normal (numpy) -> não dá erro
    df["FLAG_PIOROU_INDE"] = np.where(df["DELTA_INDE"] < 0, 1, 0).astype("int64")

# Faixa de INDE (ajustável depois)
if "INDE" in df.columns:
    df["INDE"] = pd.to_numeric(df["INDE"], errors="coerce")

    df["FAIXA_INDE"] = pd.cut(
        df["INDE"],
        bins=[-np.inf, 6.0, 7.0, 8.0, np.inf],
        labels=["Abaixo de 6", "6 a 7", "7 a 8", "Acima de 8"]
    ).astype("string")

display(df[[ID_COL, "ANO", "INDE", "FAIXA_INDE", "DELTA_INDE", "FLAG_PIOROU_INDE"]].head(20))


,RA,ANO,INDE,FAIXA_INDE,DELTA_INDE,FLAG_PIOROU_INDE
0,RA-1,2022,5.783,Abaixo de 6,0.0,0
1855,RA-1,2023,<NA>,<NA>,0.0,0
2958,RA-1,2024,<NA>,<NA>,0.0,0
9,RA-10,2022,5.784,Abaixo de 6,0.0,0
99,RA-100,2022,7.618,7 a 8,0.0,0
1072,RA-1000,2023,7.9162,7 a 8,0.0,0
2225,RA-1000,2024,8.364791,Acima de 8,0.448591,0
1074,RA-1001,2023,8.1162,Acima de 8,0.0,0
2227,RA-1001,2024,7.796432,7 a 8,-0.319768,1
1075,RA-1002,2023,7.9012,7 a 8,0.0,0


In [ ]:
# ------------------------------------------------------------
# 7) Dimensões (star schema para Power BI)
# ------------------------------------------------------------

# DIM_ALUNO: 1 linha por RA
cols_aluno = [c for c in ["RA", "NOME", "GENERO"] if c in df.columns]
dim_aluno = (
    df[cols_aluno]
    .drop_duplicates(subset=["RA"])
    .sort_values("RA")
    .reset_index(drop=True)
)

print("dim_aluno:", dim_aluno.shape)
display(dim_aluno.head())


dim_aluno: (1661, 3)


,RA,NOME,GENERO
0,RA-1,Aluno-1,Feminino
1,RA-10,Aluno-10,Feminino
2,RA-100,Aluno-100,Feminino
3,RA-1000,<NA>,Feminino
4,RA-1001,<NA>,Feminino


In [ ]:
# DIM_ESCOLA: 1 linha por instituição
if "INSTITUICAO_DE_ENSINO" in df.columns:
    dim_escola = (
        df[["INSTITUICAO_DE_ENSINO"]]
        .drop_duplicates()
        .sort_values("INSTITUICAO_DE_ENSINO")
        .reset_index(drop=True)
    )
    dim_escola["ID_ESCOLA"] = np.arange(1, len(dim_escola) + 1)

    print("dim_escola:", dim_escola.shape)
    display(dim_escola.head())
else:
    dim_escola = None


dim_escola: (13, 2)


,INSTITUICAO_DE_ENSINO,ID_ESCOLA
0,Bolsista Universitário *Formado (a),1
1,Concluiu o 3º EM,2
2,Escola JP II,3
3,Escola Pública,4
4,Nenhuma das opções acima,5


In [ ]:
# DIM_TEMPO: 1 linha por ano
dim_tempo = (
    df[["ANO"]]
    .drop_duplicates()
    .sort_values("ANO")
    .reset_index(drop=True)
)

print("dim_tempo:", dim_tempo.shape)
display(dim_tempo)


dim_tempo: (3, 1)


,ANO
0,2022
1,2023
2,2024


In [ ]:
# ------------------------------------------------------------
# 8) Criar ID_ESCOLA na fato (para relacionamento com dim_escola)
# ------------------------------------------------------------

if dim_escola is not None and "INSTITUICAO_DE_ENSINO" in df.columns:
    df = df.merge(dim_escola, on="INSTITUICAO_DE_ENSINO", how="left")

display(df[[ID_COL, "ANO", "INSTITUICAO_DE_ENSINO", "ID_ESCOLA"]].head(10))


,RA,ANO,INSTITUICAO_DE_ENSINO,ID_ESCOLA
0,RA-1,2022,Escola Pública,4
1,RA-1,2023,Privada *Parcerias com Bolsa 100%,7
2,RA-1,2024,Privada *Parcerias com Bolsa 100%,7
3,RA-10,2022,Escola Pública,4
4,RA-100,2022,Rede Decisão,12
5,RA-1000,2023,Pública,11
6,RA-1000,2024,Pública,11
7,RA-1001,2023,Pública,11
8,RA-1001,2024,Pública,11
9,RA-1002,2023,Pública,11


In [ ]:
# ------------------------------------------------------------
# 9) Fato final (fato_aluno_ano)
# ------------------------------------------------------------

# define lista de colunas que queremos na fato
cols_fato = [
    "RA", "ANO", "ID_ESCOLA",
    "FASE", "TURMA",
    "INDE", "IAA", "IAN", "IDA", "IEG", "IPP", "IPS", "IPV",
    "DELTA_INDE", "FLAG_PIOROU_INDE", "FAIXA_INDE"
]

cols_fato = [c for c in cols_fato if c in df.columns]

fato_aluno_ano = df[cols_fato].copy()

print("fato_aluno_ano:", fato_aluno_ano.shape)
display(fato_aluno_ano.head())


fato_aluno_ano: (3030, 15)


,RA,ANO,ID_ESCOLA,FASE,TURMA,INDE,IAA,IAN,IEG,IPP,IPS,IPV,DELTA_INDE,FLAG_PIOROU_INDE,FAIXA_INDE
0,RA-1,2022,4,7,A,5.783,8.3,5.0,4.1,NaN,5.6,7.278,0.0,0,Abaixo de 6
1,RA-1,2023,7,FASE 8,8E,<NA>,NaN,10.0,NaN,NaN,NaN,NaN,0.0,0,<NA>
2,RA-1,2024,7,8E,8E,<NA>,NaN,10.0,0.0,NaN,NaN,NaN,0.0,0,<NA>
3,RA-10,2022,4,7,A,5.784,8.3,5.0,5.2,NaN,5.0,7.056,0.0,0,Abaixo de 6
4,RA-100,2022,12,4,A,7.618,8.8,10.0,7.8,NaN,5.0,7.250,0.0,0,7 a 8


In [ ]:
# ------------------------------------------------------------
# 10) Exportando CSVs finais para consumir no Power BI
# ------------------------------------------------------------

fato_aluno_ano.to_csv("pede_bi_fato_aluno_ano.csv", index=False)
dim_aluno.to_csv("pede_bi_dim_aluno.csv", index=False)
dim_tempo.to_csv("pede_bi_dim_tempo.csv", index=False)

if dim_escola is not None:
    dim_escola.to_csv("pede_bi_dim_escola.csv", index=False)

print("✅ Arquivos exportados:")
print("- pede_bi_fato_aluno_ano.csv")
print("- pede_bi_dim_aluno.csv")
print("- pede_bi_dim_tempo.csv")
if dim_escola is not None:
    print("- pede_bi_dim_escola.csv")


✅ Arquivos exportados:
- pede_bi_fato_aluno_ano.csv
- pede_bi_dim_aluno.csv
- pede_bi_dim_tempo.csv
- pede_bi_dim_escola.csv
